# Module X1: exercises  (solutions)

These use `longitudinal_scores.csv`: 80 students, up to 5 waves, two groups (programme vs control).

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**ICC by hand.** Write `icc_balanced(df, subject, value)` for a **balanced** design: one-way ANOVA mean squares with the subject as the factor,
$\text{MSB}=\frac{k\sum(\bar y_i-\bar y)^2}{n-1}$, $\text{MSW}=\frac{\sum\sum(y_{ij}-\bar y_i)^2}{n(k-1)}$, and $\widehat{\text{ICC}}=\frac{\text{MSB}-\text{MSW}}{\text{MSB}+(k-1)\text{MSW}}$.
Apply it to waves 0–2 (complete for every student) as `icc`.

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

long = pd.read_csv(DATA / "longitudinal_scores.csv")

def icc_balanced(df, subject, value):
    groups = [g[value].to_numpy() for _, g in df.groupby(subject)]
    n, k = len(groups), len(groups[0])
    grand = np.concatenate(groups).mean()
    msb = k * sum((g.mean() - grand) ** 2 for g in groups) / (n - 1)
    msw = sum(((g - g.mean()) ** 2).sum() for g in groups) / (n * (k - 1))
    return (msb - msw) / (msb + (k - 1) * msw)

icc = icc_balanced(long[long["wave"] <= 2], "student_id", "score")

In [ ]:
probe = pd.DataFrame({"s": np.repeat(np.arange(30), 4), "y": np.repeat(np.random.default_rng(0).normal(0, 3, 30), 4) + np.random.default_rng(1).normal(0, 1, 120)})
assert 0.8 < icc_balanced(probe, "s", "y") < 0.99, "mostly between-subject variance"
identical = pd.DataFrame({"s": np.repeat(np.arange(10), 3), "y": np.repeat(np.arange(10.0), 3)})
assert abs(icc_balanced(identical, "s", "y") - 1) < 1e-9, "no within-subject variation gives ICC = 1"
assert abs(icc - 0.769) < 0.002, icc
print(f"✅ Correct! ICC = {icc:.3f}: about {icc:.0%} of the variation is between students")

## Exercise 2

**Design effect.** Write `design_effect(m, icc)` $=1+(m-1)\,\text{ICC}$ and `effective_n(n_total, m, icc)` $=n_{\text{total}}/\text{DEFF}$.
Compute `deff` for 5 measurements per student with the ICC from Exercise 1, and `n_eff` for the 400 nominal rows.

In [ ]:
def design_effect(m, icc):
    return 1 + (m - 1) * icc

def effective_n(n_total, m, icc):
    return n_total / design_effect(m, icc)

deff = design_effect(5, icc)
n_eff = effective_n(400, 5, icc)

In [ ]:
assert design_effect(5, 0) == 1 and design_effect(5, 1) == 5, "ICC 0 means independent rows; ICC 1 means each cluster is one observation"
assert abs(effective_n(100, 4, 0.5) - 40) < 1e-9
assert abs(deff - 4.08) < 0.01 and abs(n_eff - 98) < 1
print(f"✅ Correct! DEFF = {deff:.2f}; 400 clustered rows carry the information of about {n_eff:.0f} independent ones")

## Exercise 3

**Summary measures.** Write `student_slopes(df)` returning a DataFrame with one row per student (columns `student_id`, `group`, `slope`) where `slope` comes from `np.polyfit(wave, score, 1)`.
Then compare the groups' slopes with Welch's t-test: store `slope_diff` (programme minus control) and its 95% CI as `slope_ci` (a tuple).

In [ ]:
def student_slopes(df):
    rows = [(sid, d["group"].iloc[0], np.polyfit(d["wave"], d["score"], 1)[0]) for sid, d in df.groupby("student_id")]
    return pd.DataFrame(rows, columns=["student_id", "group", "slope"])

slopes = student_slopes(long)
a = slopes.loc[slopes["group"] == "programme", "slope"]
b = slopes.loc[slopes["group"] == "control", "slope"]
res = stats.ttest_ind(a, b, equal_var=False)
ci = res.confidence_interval(0.95)
slope_diff = a.mean() - b.mean()
slope_ci = (ci.low, ci.high)

In [ ]:
assert list(slopes.columns) == ["student_id", "group", "slope"] and len(slopes) == 80
assert abs(slope_diff - 1.87) < 0.01, slope_diff
assert abs(slope_ci[0] - 1.058) < 0.01 and abs(slope_ci[1] - 2.683) < 0.01, slope_ci
print(f"✅ Correct! Programme students gain {slope_diff:.2f} more points per wave (95% CI {slope_ci[0]:.2f} to {slope_ci[1]:.2f})")

## Exercise 4

**A mixed model.** Fit `score ~ wave * group` with a random intercept **and random slope for wave** per student (`smf.mixedlm(..., groups=long["student_id"], re_formula="~wave").fit()`),
as `model`. Store the interaction estimate (`wave:group[T.programme]`) as `inter_est` and its 95% CI as `inter_ci` (a tuple).

In [ ]:
model = smf.mixedlm("score ~ wave * group", data=long, groups=long["student_id"], re_formula="~wave").fit()
term = "wave:group[T.programme]"
inter_est = model.params[term]
inter_ci = tuple(model.conf_int().loc[term])

In [ ]:
assert abs(inter_est - 1.859) < 0.02, inter_est
assert abs(inter_ci[0] - 1.083) < 0.03 and abs(inter_ci[1] - 2.635) < 0.03, inter_ci
assert abs(inter_est - slope_diff) < 0.1, "the summary-measures answer and the mixed-model answer should agree closely"
print(f"✅ Correct! Interaction = {inter_est:.2f} (95% CI {inter_ci[0]:.2f} to {inter_ci[1]:.2f})")

## Exercise 5

**Naive vs cluster-robust.** Fit the ordinary regression `score ~ wave * group` (`smf.ols`) twice: once with default standard errors (`ols_naive`) and once with `cov_type="cluster", cov_kwds={"groups": long["student_id"]}` (`ols_cluster`).
Store the standard errors of the interaction term as `se_naive` and `se_cluster`.

In [ ]:
ols_naive = smf.ols("score ~ wave * group", long).fit()
ols_cluster = smf.ols("score ~ wave * group", long).fit(cov_type="cluster", cov_kwds={"groups": long["student_id"]})
se_naive = ols_naive.bse["wave:group[T.programme]"]
se_cluster = ols_cluster.bse["wave:group[T.programme]"]

In [ ]:
assert abs(ols_naive.params["wave:group[T.programme]"] - ols_cluster.params["wave:group[T.programme]"]) < 1e-9, "the estimates are the same; only the standard errors change"
assert abs(se_naive - 0.685) < 0.005 and abs(se_cluster - 0.409) < 0.005, (se_naive, se_cluster)
assert se_naive > se_cluster, "for a within-student effect the naive SE is too large"
print(f"✅ Correct! naive SE {se_naive:.3f} vs cluster-robust {se_cluster:.3f}")

## Exercise 6

**Random intercept or random slope?** Fit both mixed models by **maximum likelihood** (`.fit(reml=False)`; wrap in `warnings.catch_warnings()` if convergence warnings appear): intercept only (`ml_int`) and intercept + slope (`ml_slope`).
Compute `aic_int` and `aic_slope` with $\text{AIC}=-2\,\ell+2\,(p_{\text{fixed}}+p_{\text{cov}}+1)$, where `p_cov` is 1 for the intercept-only model and 3 for the intercept + slope model
(variance of the intercept; plus variance of the slope and their covariance), and `best` = the name of the better model (`"intercept"` or `"intercept+slope"`).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ml_int = smf.mixedlm("score ~ wave * group", long, groups=long["student_id"]).fit(reml=False)
    ml_slope = smf.mixedlm("score ~ wave * group", long, groups=long["student_id"], re_formula="~wave").fit(reml=False)
p_fixed = len(ml_int.fe_params)
aic_int = -2 * ml_int.llf + 2 * (p_fixed + 1 + 1)
aic_slope = -2 * ml_slope.llf + 2 * (p_fixed + 3 + 1)
best = "intercept+slope" if aic_slope < aic_int else "intercept"

In [ ]:
assert abs(aic_int - 2415.5) < 1.0 and abs(aic_slope - 2398.2) < 1.0, (aic_int, aic_slope)
assert best == "intercept+slope"
print(f"✅ Correct! AIC {aic_int:.1f} (intercept) vs {aic_slope:.1f} (intercept + slope): allow students to differ in growth rate")

## Exercise 7

**Missing waves.** Compute `n_complete`: the number of students with all 5 waves, and `share_missing`: the fraction of the 400 possible student-wave rows that are missing.

In [ ]:
per_student = long.groupby("student_id").size()
n_complete = int((per_student == 5).sum())
share_missing = 1 - len(long) / (80 * 5)

In [ ]:
assert n_complete == 61, n_complete
assert abs(share_missing - 0.0475) < 1e-9, share_missing
print(f"✅ Correct! {n_complete} of 80 students have every wave ({share_missing:.2%} of rows missing). A repeated-measures ANOVA would discard the other {80 - n_complete} students.")

## Exercise 8

**Choose the analysis.** Match each design to the most appropriate analysis. Fill `answers` with one of `"paired_t"`, `"independent_t"`, `"mixed_model"`, `"summary_slopes"`, `"cluster_robust"`.

* **a)** The same 30 patients' blood pressure before and after one treatment (2 time points).
* **b)** Test scores of students sampled from 25 classrooms; you want a regression on student-level predictors and classrooms differ.
* **c)** Weights of 12 patients measured monthly for a year, some visits missing, with a time-varying covariate (dose).
* **d)** 60 independent people randomly split into two groups and measured once.
* **e)** You want a quick, transparent comparison of how fast two groups improve over 5 waves with a complete, balanced design.

In [ ]:
answers = {"a": "paired_t", "b": "mixed_model", "c": "mixed_model", "d": "independent_t", "e": "summary_slopes"}

In [ ]:
expected = {"a": "paired_t", "b": "mixed_model", "c": "mixed_model", "d": "independent_t", "e": "summary_slopes"}
for key, want in expected.items():
    assert answers[key] == want, f"({key}) think about what is repeated or clustered, and whether data are unbalanced"
print("✅ Correct!")